<a href="https://colab.research.google.com/github/BreakTheAlgorithm/MLforALL/blob/main/book_ch14_neural_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>🧠 Chapter 14 — Introduction to Neural Networks</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 80 mins | Level: Intermediate–Advanced</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Understand the perceptron, activation functions, and MLP architecture
- Implement forward pass in NumPy
- Understand backpropagation at a conceptual level
- Use sklearn's MLPClassifier for a practical classification task
- Explain vanishing gradients and the role of activation functions

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network  import MLPClassifier
from sklearn.datasets        import make_circles, make_classification
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics         import accuracy_score, classification_report

%matplotlib inline
np.random.seed(42)

## 📖 Section A — Activation Functions

| Function | Formula | Range | Used Where |
|----------|---------|-------|--------|
| Sigmoid  | $\sigma(x)=\frac{1}{1+e^{-x}}$ | (0,1) | Binary output |
| ReLU     | $\max(0, x)$ | [0,∞) | Hidden layers (default) |
| Tanh     | $\frac{e^x-e^{-x}}{e^x+e^{-x}}$ | (-1,1) | Hidden layers |
| Softmax  | $\frac{e^{x_i}}{\sum e^{x_j}}$ | (0,1)→sums to 1 | Multi-class output |

> 💡 **Key Rule:** ReLU is the default for hidden layers. Sigmoid/Softmax go in the OUTPUT layer. Never use sigmoid in deep hidden layers — it causes vanishing gradients (very small gradients kill learning in early layers).

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: Activation Functions
# ─────────────────────────────────────────────────────────────
x = np.linspace(-5, 5, 200)
funcs = {
    'Sigmoid': 1 / (1 + np.exp(-x)),
    'ReLU':    np.maximum(0, x),
    'Tanh':    np.tanh(x),
}

plt.figure(figsize=(10, 4))
for name, vals in funcs.items():
    plt.plot(x, vals, label=name, linewidth=2)
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.axvline(0, color='gray', linestyle='--', alpha=0.5)
plt.legend()
plt.title('Activation Functions')
plt.grid(True, alpha=0.3)
plt.show()

## 📖 Section B — Neural Network Forward Pass

For a 2-layer network:
$$
Z^{[1]} = X W^{[1]} + b^{[1]}, \quad A^{[1]} = \text{ReLU}(Z^{[1]})
$$
$$
Z^{[2]} = A^{[1]} W^{[2]} + b^{[2]}, \quad \hat{y} = \sigma(Z^{[2]})
$$

```python
# Forward pass from scratch (NumPy)
def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))

Z1 = X @ W1 + b1
A1 = relu(Z1)
Z2 = A1 @ W2 + b2
y_hat = sigmoid(Z2)
```

---
## 🟢 Exercise 14.1 — Forward Pass from Scratch *(Level 1 · Est. 15 min)*

> Implement a 2-layer neural network forward pass using NumPy. Architecture: input=2, hidden=4, output=1. Use ReLU for hidden, Sigmoid for output. Initialise weights randomly.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 14.1: Forward Pass (NumPy)
# ─────────────────────────────────────────────────────────────

def relu(z):
    # YOUR CODE HERE
    pass

def sigmoid(z):
    # YOUR CODE HERE
    pass

def forward_pass(X, W1, b1, W2, b2):
    """
    X: (n, 2)
    W1: (2, 4), b1: (1, 4)
    W2: (4, 1), b2: (1, 1)
    Returns: y_hat (n, 1)
    """
    # YOUR CODE HERE
    pass

# Test
np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.1
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.1
b2 = np.zeros((1, 1))

X_test_in = np.array([[1.0, 2.0], [-1.0, 0.5], [0.0, -1.0]])
y_hat = forward_pass(X_test_in, W1, b1, W2, b2)
print('Predictions (should be between 0 and 1):', y_hat.flatten())

assert y_hat.shape == (3, 1), 'Output shape should be (3, 1)'
assert np.all((y_hat > 0) & (y_hat < 1)), 'Sigmoid output should be in (0,1)'
print('✅ Forward pass correct!')

In [ ]:
# @title ✅ Solution — Exercise 14.1 (click to expand)

def relu(z):    return np.maximum(0, z)
def sigmoid(z): return 1 / (1 + np.exp(-z))

def forward_pass(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1       # (n, 4)
    A1 = relu(Z1)           # (n, 4)
    Z2 = A1 @ W2 + b2      # (n, 1)
    y_hat = sigmoid(Z2)     # (n, 1)
    return y_hat

np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.1
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.1
b2 = np.zeros((1, 1))

X_test_in = np.array([[1.0, 2.0], [-1.0, 0.5], [0.0, -1.0]])
y_hat = forward_pass(X_test_in, W1, b1, W2, b2)
print('Predictions:', y_hat.flatten())
print('✅ Matrix dimensions: X(n,2) @ W1(2,4) = Z1(n,4) → A1(n,4) @ W2(4,1) = Z2(n,1) → y_hat(n,1)')

---
## 🟡 Exercise 14.2 — MLP on XOR and Circle Datasets *(Level 2 · Est. 20 min)*

> Show that linear models fail on the circle dataset but MLP succeeds. Compare Logistic Regression vs MLP accuracy. Visualise decision boundaries.

In [ ]:
# @title ✅ Solution — Exercise 14.2 (click to expand)
from sklearn.linear_model import LogisticRegression
from sklearn.inspection   import DecisionBoundaryDisplay

X_circ, y_circ = make_circles(n_samples=400, noise=0.1, random_state=42, factor=0.3)
scaler = StandardScaler()
X_circ_s = scaler.fit_transform(X_circ)

lr  = LogisticRegression()
mlp = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', max_iter=1000, random_state=42)

lr.fit(X_circ_s, y_circ)
mlp.fit(X_circ_s, y_circ)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, model, title in [(axes[0], lr, f'Logistic Regression ({accuracy_score(y_circ, lr.predict(X_circ_s)):.2%})'),
                          (axes[1], mlp, f'MLP (16,8) ({accuracy_score(y_circ, mlp.predict(X_circ_s)):.2%})')]:
    DecisionBoundaryDisplay.from_estimator(model, X_circ_s, ax=ax, alpha=0.3, cmap='RdBu')
    ax.scatter(X_circ_s[:, 0], X_circ_s[:, 1], c=y_circ, cmap='RdBu', edgecolors='k', s=20)
    ax.set_title(title)

plt.suptitle('Linear vs Non-Linear Decision Boundary', fontsize=12)
plt.tight_layout()
plt.show()
print('✅ LR can only draw a LINE boundary — it cannot separate concentric circles.')
print('   MLP learns a curved boundary by composing non-linear activations.')

---
## 🔴 Exercise 14.3 — Effect of Hidden Layer Architecture *(Level 3 · Est. 20 min)*

> Compare MLPClassifier with different architectures on make_classification (1000 samples, 20 features). Report accuracy and training loss. Plot training loss curve for each.

In [ ]:
# @title ✅ Solution — Exercise 14.3 (click to expand)

X_mc, y_mc = make_classification(n_samples=1000, n_features=20, n_informative=10,
                                  random_state=42)
scaler2 = StandardScaler()
X_mc_s = scaler2.fit_transform(X_mc)
X_tr, X_te, y_tr, y_te = train_test_split(X_mc_s, y_mc, test_size=0.2, random_state=42)

architectures = {
    'Shallow (32,)':     (32,),
    'Wide (128,)':       (128,),
    'Deep (64, 32, 16)': (64, 32, 16),
    'Deep (128, 64, 32)':(128, 64, 32),
}

results = []
fig, ax = plt.subplots(figsize=(9, 5))

for name, layers in architectures.items():
    mlp = MLPClassifier(hidden_layer_sizes=layers, activation='relu',
                        max_iter=200, random_state=42, early_stopping=False)
    mlp.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mlp.predict(X_te))
    params = sum(np.prod(c.shape) for c in mlp.coefs_) + sum(len(b) for b in mlp.intercepts_)
    results.append({'Architecture': name, 'Accuracy': round(acc, 4), 'Parameters': params, 'Iters': mlp.n_iter_})
    ax.plot(mlp.loss_curve_, label=name)

ax.set_title('Training Loss Curve by Architecture')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print('\n✅ Deeper networks can capture more complex patterns but may need more training epochs.')
print('Observe: loss curves show which architecture converges faster and to lower loss.')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: What is backpropagation?</strong></summary>

**Answer:** Backpropagation is the algorithm that computes gradients of the loss function with respect to every weight in the network. It applies the chain rule of calculus backwards from the output layer to the input layer. For each weight, we compute ∂L/∂w — how much the loss changes with a tiny change in that weight. These gradients are then used by gradient descent to update: w ← w − α·∂L/∂w. Without backprop, training deep networks would require computing gradients manually — computationally infeasible for millions of parameters.
</details>

<details>
<summary><strong>Q2: What is the vanishing gradient problem?</strong></summary>

**Answer:** In deep networks with sigmoid or tanh activations, gradients can become extremely small (close to 0) as they propagate backwards. This is because the derivative of sigmoid is at most 0.25 — multiply this across 10 layers and the gradient at layer 1 is (0.25)^10 ≈ 0.000001. Weights in early layers barely update, so the network can't learn. Solutions: (1) Use ReLU — derivative is 1 for positive inputs, so gradients don't vanish. (2) Batch normalisation. (3) Residual connections (skip connections in ResNet).
</details>

<details>
<summary><strong>Q3: How do you choose the number of hidden layers and neurons?</strong></summary>

**Answer:** No universal rule — it's empirical. Guidelines: (1) Start with 1-2 hidden layers — sufficient for most tabular ML problems. (2) Each layer with roughly half the neurons of the previous (funnel shape). (3) More neurons = more capacity but higher risk of overfitting. (4) Use cross-validation to compare architectures. (5) Add dropout or regularisation if overfitting. Deep learning (>5 layers) is needed for images, audio, NLP — not for tabular data.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 14 Complete!</h3>
<ul>
<li>Activation functions: Sigmoid, ReLU, Tanh</li>
<li>Forward pass implementation in NumPy</li>
<li>MLP vs linear models on non-linear datasets</li>
<li>Architecture comparison and training curves</li>
</ul>
<p><strong>Next:</strong> Chapter 15 — Deep Learning with Keras and TensorFlow</p>
</div>